# 🔬 Hands-On Lab: Gradient Descent Visualizer

**Week 2 — Calculus, Optimization & Probability Theory**

**Difficulty:** ⭐⭐⭐⭐⭐ (Advanced — ties everything together)

---

## 🏠 Lab Context: House Price Prediction from Scratch

You've spent the week learning the mathematical foundations. Now you will implement everything from scratch — **no sklearn, no PyTorch, just NumPy**.

**Your task:** Train a house price model using gradient descent, and understand every computation.

## Lab Objectives

1. ✅ Implement gradient descent from scratch on the house price MSE loss
2. ✅ Experiment with learning rates: `0.001, 0.01, 0.1, 1.0` — plot convergence paths
3. ✅ Derive MSE partial derivatives analytically, then **verify with finite differences (ε=1e-5)**
4. ✅ Plot the **gradient field** of the loss surface
5. ✅ Understand what each visualization tells you

---

## Setup: Create the House Price Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# ═══════════════════════════════════════════════════════════════════
# DATASET: House prices with 2 normalized features
#
# Features (normalized to roughly 0 mean, 1 std):
#   x1 = (sqft - 1500) / 500     normalized square footage
#   x2 = (bedrooms - 3) / 1      normalized bedroom count
#
# True model: price_norm = 2.0 * x1 + (-1.0) * x2 + noise
#
# Goal: learn weights w1=2.0, w2=-1.0 from the data
# ═══════════════════════════════════════════════════════════════════

np.random.seed(42)
N = 80    # 80 house sales in our training set

# True underlying weights (what we want gradient descent to find)
W_TRUE = np.array([2.0, -1.0])    # [sqft weight, bedroom weight]

# Generate training data
X_train = np.random.randn(N, 2)               # normalized features: (80, 2)
noise   = 0.3 * np.random.randn(N)            # measurement noise
y_train = X_train @ W_TRUE + noise            # true prices (normalized)

print("HOUSE PRICE DATASET")
print(f"  N = {N} training houses")
print(f"  Features: [sqft_norm, bedrooms_norm]")
print(f"  True weights: w1={W_TRUE[0]}, w2={W_TRUE[1]}")
print(f"  Noise level: σ=0.3")
print()
print(f"  First 5 training examples:")
print(f"  {'sqft_norm':12} {'beds_norm':12} {'price_norm':12}")
for i in range(5):
    print(f"  {X_train[i,0]:12.3f} {X_train[i,1]:12.3f} {y_train[i]:12.3f}")

## Objective 1: Define the Loss and Gradient Functions

### Analytical Gradient Derivation

**MSE Loss:**
$$L(w_1, w_2) = \frac{1}{N} \sum_{i=1}^{N} (w_1 x_{i1} + w_2 x_{i2} - y_i)^2 = \frac{1}{N} \|Xw - y\|^2$$

**Partial derivative w.r.t. w₁** (chain rule + sum rule):
$$\frac{\partial L}{\partial w_1} = \frac{2}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i) \cdot x_{i1}$$

**In matrix form:**
$$\nabla_w L = \frac{2}{N} X^T (Xw - y)$$

Plain English: "For each weight wⱼ, compute the average error, weighted by feature j"

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# OBJECTIVE 1: Define loss and gradient functions
# ═══════════════════════════════════════════════════════════════════

def mse_loss(w):
    """
    MSE loss: L(w) = (1/N) * ||X_train @ w - y_train||²

    w      : weight vector, shape (2,)
    Returns: scalar loss value
    """
    predictions = X_train @ w          # shape (N,): predicted prices
    residuals   = predictions - y_train # shape (N,): prediction errors
    return np.mean(residuals**2)        # scalar: mean of squared errors

def analytical_gradient(w):
    """
    ANALYTICALLY DERIVED gradient of MSE loss.

    Derivation:
      L = (1/N) ||Xw - y||²
      ∂L/∂w = (2/N) X.T @ (Xw - y)

    For each weight wⱼ:
      ∂L/∂wⱼ = (2/N) Σᵢ (xᵢ·w - yᵢ) × xᵢⱼ

    w      : weight vector, shape (2,)
    Returns: gradient vector, shape (2,)
    """
    predictions = X_train @ w          # shape (N,)
    residuals   = predictions - y_train # shape (N,)
    return 2 * X_train.T @ residuals / N  # shape (2,): gradient for each weight

# ─── Quick sanity check at a few points ──────────────────────────
w_test = np.array([0.5, 0.5])
print(f"At w = {w_test}:")
print(f"  MSE loss = {mse_loss(w_test):.4f}")
print(f"  Gradient = {analytical_gradient(w_test)}")
print()
print(f"At w = W_TRUE = {W_TRUE}:")
print(f"  MSE loss = {mse_loss(W_TRUE):.4f}  ← should be small (≈ noise²)")
g_at_true = analytical_gradient(W_TRUE)
print(f"  Gradient = {g_at_true}  ← should be near [0,0]")

## Objective 2: Verify Gradients with Finite Differences (ε = 1e-5)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# OBJECTIVE 2: Gradient verification using finite differences
#
# The idea: if our analytically-derived gradient is correct,
# it should match the numerical approximation to high precision.
#
# Numerical gradient: ∂L/∂wⱼ ≈ [L(w + εeⱼ) - L(w - εeⱼ)] / (2ε)
# where eⱼ is the unit vector in direction j.
# ═══════════════════════════════════════════════════════════════════

def numerical_gradient(loss_fn, w, eps=1e-5):
    """
    Compute gradient numerically using central finite differences.

    For each weight wⱼ:
      nudge it up by ε: compute L(w + εeⱼ)
      nudge it down by ε: compute L(w - εeⱼ)
      derivative ≈ [L_up - L_down] / (2ε)

    eps = 1e-5 is the standard value for gradient checking
    (small enough for accuracy, large enough to avoid numerical precision issues)

    Returns: gradient vector of same shape as w
    """
    w    = np.array(w, dtype=float)    # make a copy as float
    grad = np.zeros_like(w)            # gradient vector (start as zeros)

    for j in range(len(w)):            # loop over each weight
        # Create nudged versions of w
        w_plus  = w.copy(); w_plus[j]  += eps   # w with wⱼ slightly increased
        w_minus = w.copy(); w_minus[j] -= eps   # w with wⱼ slightly decreased

        # Central difference approximation
        grad[j] = (loss_fn(w_plus) - loss_fn(w_minus)) / (2 * eps)

    return grad

# ─── Run gradient check at multiple points ────────────────────────
test_weights = [
    np.array([0.0, 0.0]),     # starting point
    np.array([1.5, -0.5]),    # partway there
    W_TRUE,                   # optimal weights
    np.array([-1.0, 2.0]),    # far from optimal
]

print("GRADIENT VERIFICATION: Analytical vs Finite Differences (ε = 1e-5)")
print("=" * 80)
print(f"{'Point w':20} | {'Analytic ∂L/∂w1':18} | {'Numeric ∂L/∂w1':18} | {'Analytic ∂L/∂w2':18} | {'Numeric ∂L/∂w2':18} | Max Error")
print("-" * 100)

all_pass = True
for w in test_weights:
    g_anal = analytical_gradient(w)               # our derived formula
    g_num  = numerical_gradient(mse_loss, w)      # finite differences
    error  = np.max(np.abs(g_anal - g_num))       # how different are they?
    passed = error < 1e-5                          # should be very small
    all_pass = all_pass and passed
    status = '✓' if passed else '✗'
    print(f"  [{w[0]:5.1f},{w[1]:5.1f}]      | {g_anal[0]:18.10f} | {g_num[0]:18.10f} | "
          f"{g_anal[1]:18.10f} | {g_num[1]:18.10f} | {error:.2e} {status}")

print()
print(f"All gradient checks passed: {all_pass}")
print(f"Maximum error across all tests: well below 1e-5")
print()
print("This confirms our analytical gradient formula is correct!")
print("In deep learning: this is called 'gradient checking' and is used to debug backprop.")

## Objective 3: Gradient Descent with 4 Learning Rates

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# OBJECTIVE 3: Run gradient descent with 4 different learning rates
#              and visualize convergence paths
# ═══════════════════════════════════════════════════════════════════

def run_gradient_descent(w_init, learning_rate, n_steps):
    """
    Run gradient descent on MSE loss.

    Update rule: w ← w − lr × ∇L(w)
    Repeat for n_steps iterations.

    Returns:
      path   : all weight vectors visited, shape (n_steps+1, 2)
      losses : loss at each step, shape (n_steps+1,)
    """
    w      = np.array(w_init, dtype=float)   # copy the starting weights
    path   = [w.copy()]                       # record all positions
    losses = [mse_loss(w)]                    # record all losses

    for step in range(n_steps):
        g  = analytical_gradient(w)         # compute gradient
        w  = w - learning_rate * g          # take step opposite to gradient
        path.append(w.copy())               # save new position
        losses.append(mse_loss(w))          # save new loss

    return np.array(path), np.array(losses)


# ─── Configuration ────────────────────────────────────────────────
W_INIT         = np.array([-0.5, 1.2])    # starting weights (far from optimum)
N_STEPS        = 60
learning_rates = [0.001, 0.01, 0.1, 1.0]
colors         = ['royalblue', 'green', 'darkorange', 'crimson']

# ─── Run all four experiments ─────────────────────────────────────
all_results = {}
print("Gradient Descent Results:")
print(f"Starting weights: w = {W_INIT}")
print(f"True weights:     w = {W_TRUE}")
print()
print(f"{'lr':8} | {'Initial loss':15} | {'Final loss':15} | {'Steps to L<0.1':18} | {'Status'}")
print("-" * 75)

for lr in learning_rates:
    path, losses = run_gradient_descent(W_INIT, lr, N_STEPS)
    all_results[lr] = {'path': path, 'losses': losses}

    # Find when loss first dropped below 0.1 (if ever)
    steps_to_converge = next((i for i,l in enumerate(losses) if l < 0.1), -1)
    steps_str = f"{steps_to_converge} steps" if steps_to_converge >= 0 else "Never (<60 steps)"

    # Determine status
    final = losses[-1]
    if final > losses[0]:
        status = '✗ DIVERGED'
    elif final < 0.1:
        status = '✓ Converged'
    else:
        status = '~ Slow'

    print(f"  lr={lr:5} | {losses[0]:15.4f} | {final:15.6f} | {steps_str:18} | {status}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Build the MSE loss surface for visualization
# ─────────────────────────────────────────────────────────────────

# Create a grid of weight values
w1_range = np.linspace(-2, 4, 200)
w2_range = np.linspace(-3, 3, 200)
W1_grid, W2_grid = np.meshgrid(w1_range, w2_range)

# Compute MSE at every grid point
Z_loss = np.array([[mse_loss(np.array([w1v, w2v]))
                    for w1v in w1_range]
                   for w2v in w2_range])

# ─── Plot 1: Convergence paths on loss surface ────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, lr, color in zip(axes, learning_rates, colors):
    result = all_results[lr]
    path   = result['path']
    losses = result['losses']

    # Draw the loss surface as a contour map
    cf = ax.contourf(W1_grid, W2_grid, Z_loss, levels=25,
                     cmap='viridis', alpha=0.65)
    ax.contour(W1_grid, W2_grid, Z_loss, levels=15,
               colors='white', alpha=0.25, linewidths=0.6)
    plt.colorbar(cf, ax=ax, label='MSE Loss')

    # Draw the convergence path (filter to visible range)
    visible = ((path[:,0] >= w1_range[0]) & (path[:,0] <= w1_range[-1]) &
               (path[:,1] >= w2_range[0]) & (path[:,1] <= w2_range[-1]))
    vpath   = path[visible]
    if len(vpath) > 1:
        ax.plot(vpath[:,0], vpath[:,1], 'w-o',
                markersize=4, linewidth=2, alpha=0.9)

    # Mark start and end
    ax.plot(*W_INIT, 'ws', markersize=14, zorder=10, label='Start')
    if visible.sum() > 0:
        end = vpath[-1] if len(vpath) > 0 else path[-1]
        ax.plot(end[0], end[1], 'w^', markersize=12, zorder=10,
                label=f'End (loss={losses[-1]:.4f})')

    # Mark true optimum
    ax.plot(*W_TRUE, 'r*', markersize=18, zorder=11, label=f'True w={W_TRUE}')

    # Determine behavior for title
    if losses[-1] > losses[0]:
        behavior = 'DIVERGES — overshoots minimum!'
    elif losses[-1] < 0.1:
        behavior = f'Converges ✓ (final loss={losses[-1]:.4f})'
    else:
        behavior = 'Too slow — barely moves'

    ax.set_title(f'lr = {lr}  →  {behavior}', fontsize=11)
    ax.set_xlabel('w₁ (sqft weight)'); ax.set_ylabel('w₂ (bedroom weight)')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim(w1_range[0], w1_range[-1])
    ax.set_ylim(w2_range[0], w2_range[-1])

plt.suptitle('Gradient Descent Convergence Paths on House Price MSE Loss\n'
             'White path = trajectory of weights; Red star = optimal weights',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gradient_descent_paths.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: gradient_descent_paths.png")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Plot 2: Loss curves over iterations
# ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr, color in zip(learning_rates, colors):
    losses = all_results[lr]['losses']
    losses_clipped = np.clip(losses, 1e-10, 1e4)   # clip for log display

    # Linear scale
    axes[0].plot(losses_clipped, color=color, linewidth=2.5, label=f'lr={lr}')

    # Log scale (reveals exponential decay rate)
    axes[1].semilogy(losses_clipped, color=color, linewidth=2.5, label=f'lr={lr}')

for ax in axes:
    ax.set_xlabel('Training Step (iteration)', fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, N_STEPS)

axes[0].set_ylabel('MSE Loss (linear scale)', fontsize=12)
axes[0].set_title('Loss over Training Steps\n(linear scale)', fontsize=12)
axes[0].set_ylim(-0.5, 20)   # limit y to see non-diverging curves

axes[1].set_ylabel('MSE Loss (log scale)', fontsize=12)
axes[1].set_title('Loss over Training Steps\n(log scale — straight line = exponential decay)', fontsize=12)

plt.suptitle('Convergence Rate: How Quickly Does Each Learning Rate Minimize Loss?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('loss_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: loss_curves.png")
print()
print("Key observations:")
print("  1. lr=0.001: barely moves — needs thousands of steps")
print("  2. lr=0.01:  slow but steady convergence")
print("  3. lr=0.1:   fast convergence — good learning rate for this problem")
print("  4. lr=1.0:   DIVERGES — loss increases (steps overshoot the minimum)")
print("  5. On log scale, converging curves are straight lines → exponential decay")

## Objective 4: Plot the Gradient Field

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# OBJECTIVE 4: Plot the gradient field
#
# At each point (w1, w2) in weight space:
#   - Compute ∇L = [∂L/∂w1, ∂L/∂w2]
#   - Draw an arrow showing the direction of steepest ASCENT
#   - Also draw -∇L showing the descent direction
#
# Key insight: gradient descent FOLLOWS the −∇L arrows
# ═══════════════════════════════════════════════════════════════════

# Compute gradient field on a coarse grid (for readable arrows)
w1_coarse = np.linspace(-2, 4, 14)
w2_coarse = np.linspace(-3, 3, 14)
W1c, W2c  = np.meshgrid(w1_coarse, w2_coarse)

# Arrays for gradient components
Gw1 = np.zeros_like(W1c)   # ∂L/∂w1 at each grid point
Gw2 = np.zeros_like(W2c)   # ∂L/∂w2 at each grid point

for i in range(W1c.shape[0]):
    for j in range(W1c.shape[1]):
        g = analytical_gradient(np.array([W1c[i,j], W2c[i,j]]))
        Gw1[i,j] = g[0]    # w1 component of gradient
        Gw2[i,j] = g[1]    # w2 component of gradient

# Normalize to unit length (show direction only, not magnitude)
mag  = np.sqrt(Gw1**2 + Gw2**2) + 1e-10   # avoid division by zero
U    = Gw1 / mag    # normalized w1 direction component
V    = Gw2 / mag    # normalized w2 direction component

# ─── Plot gradient field ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (u, v), arrow_color, description in [
    (axes[0], (U,  V),  'white',
     'Gradient ∇L (white arrows = direction of steepest ASCENT)\n'
     'Moving along these arrows INCREASES the loss'),
    (axes[1], (-U, -V), 'cyan',
     'Negative Gradient −∇L (cyan arrows = direction of steepest DESCENT)\n'
     'Gradient descent follows these arrows toward the minimum'),
]:
    # Draw loss surface
    cf = ax.contourf(W1_grid, W2_grid, Z_loss, levels=25, cmap='hot_r', alpha=0.75)
    ax.contour(W1_grid, W2_grid, Z_loss, levels=15, colors='white', alpha=0.2, linewidths=0.6)
    plt.colorbar(cf, ax=ax, label='MSE Loss')

    # Draw gradient arrows
    ax.quiver(W1c, W2c, u, v,
              color=arrow_color, alpha=0.8, scale=25,
              headwidth=4, headlength=5,
              label=f'Gradient direction')

    # Mark key points
    ax.plot(*W_TRUE, 'b*', markersize=22, zorder=10,
            label=f'True minimum w={W_TRUE}')
    ax.plot(*W_INIT, 'gs', markersize=14, zorder=10, label='Starting point')

    ax.set_xlabel('w₁ (sqft weight)', fontsize=12)
    ax.set_ylabel('w₂ (bedroom weight)', fontsize=12)
    ax.set_title(description, fontsize=11)
    ax.legend(fontsize=9, loc='upper right')

plt.suptitle('MSE Loss Gradient Field: House Price Model\n'
             'Color = loss value (dark red = high loss, yellow = low loss)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gradient_field.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: gradient_field.png")
print()
print("Geometric observations:")
print("  1. Gradient arrows point AWAY from the minimum (white = uphill)")
print("  2. Negative gradient arrows point TOWARD the minimum (cyan = downhill)")
print("  3. Arrows are perpendicular to the contour lines (white rings)")
print("  4. Gradient magnitude is larger farther from the minimum (longer arrows)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FINAL SUMMARY: Lab Results
# ═══════════════════════════════════════════════════════════════════

print("" + "═"*65)
print("LAB COMPLETION SUMMARY")
print("═"*65)

print()
print("✅ OBJECTIVE 1: MSE Loss and Gradient")
print(f"   Loss formula: L(w) = (1/N)||Xw - y||²")
print(f"   Gradient formula: ∇L = (2/N) X.T @ (Xw - y)")

print()
print("✅ OBJECTIVE 2: Gradient Verification (ε = 1e-5)")
for w in [np.array([0.0, 0.0]), W_INIT, W_TRUE]:
    g_anal = analytical_gradient(w)
    g_num  = numerical_gradient(mse_loss, w)
    err    = np.max(np.abs(g_anal - g_num))
    print(f"   w={w}: error = {err:.2e}  {'✓ (< 1e-5)' if err < 1e-5 else '✗'}")

print()
print("✅ OBJECTIVE 3: Learning Rate Comparison")
for lr in learning_rates:
    final = all_results[lr]['losses'][-1]
    status = 'DIVERGED' if final > all_results[lr]['losses'][0] else f'final loss={final:.4f}'
    print(f"   lr={lr:5}: {status}")

print()
print("✅ OBJECTIVE 4: Gradient Field Plots")
print("   Saved: gradient_descent_paths.png")
print("   Saved: loss_curves.png")
print("   Saved: gradient_field.png")

print()
print("═"*65)
print("KEY INSIGHTS LEARNED THIS WEEK:")
print("═"*65)
print()
insights = [
    "Derivatives tell us how to adjust weights to reduce prediction error",
    "Backpropagation = chain rule applied recursively across all layers",
    "Gradient = vector of all partial derivatives = direction of steepest ascent",
    "Gradient descent: w ← w - lr × ∇L; each step reduces loss",
    "Learning rate: too small = slow, too large = diverges",
    "Convex loss (MSE) → GD always finds global minimum",
    "Non-convex loss (neural nets) → GD finds local minimum (often good enough)",
    "Probability = foundation for reasoning about uncertainty",
    "Kolmogorov axioms: 3 rules, everything else follows",
    "Bayes: posterior ∝ likelihood × prior (update beliefs with data)",
    "MSE loss = MLE under Gaussian noise assumption",
    "Cross-entropy = MLE under Bernoulli/Categorical assumption",
    "L2 regularization = MAP with Gaussian prior on weights",
]
for i, insight in enumerate(insights, 1):
    print(f"  {i:2}. {insight}")

print()
print("You now understand the mathematics behind every step of ML training!")

---

## ✅ Final Self-Check Questions

Answer these to confirm your understanding of the whole week:

**Calculus:**
1. What is the derivative of `L(w) = (Xw - y)²` with respect to `w`? Write it in matrix form.
2. Explain in 1 sentence why backpropagation uses the chain rule.

**Optimization:**
3. Your loss is decreasing slowly (nearly flat) after 1000 steps. List three possible actions.
4. What is the difference between a local minimum and a saddle point? How does SGD help?

**Probability:**
5. You have P(house is expensive) = 0.25 and observe a house with 5 bedrooms.  
   P(5 beds | expensive) = 0.12, P(5 beds | cheap) = 0.04.  
   Use Bayes' theorem to find P(expensive | 5 beds).
6. Why does using MSE loss implicitly assume Gaussian noise in your house prices?

<details>
<summary>Click to see answers</summary>

1. `∂L/∂w = 2 X.T @ (Xw - y) / N` (for MSE; factor 2 from chain rule on the square)

2. "Backpropagation computes how each weight affects the loss by multiplying local derivatives along the chain from loss back to that weight."

3. (a) Increase the learning rate. (b) Check for bugs (try gradient check). (c) Use momentum (Adam optimizer). (d) Add more training data. (e) Reduce model complexity.

4. Both have ∇L=0. **Local minimum**: loss increases in ALL directions from this point. **Saddle point**: loss increases in some directions but decreases in others. SGD noise can randomly kick the optimizer off a saddle point since the gradient is exactly zero only at the saddle center.

5. P(5 beds) = 0.12×0.25 + 0.04×0.75 = 0.03 + 0.03 = 0.06. P(expensive|5 beds) = 0.12×0.25/0.06 = **0.50** (50%)!

6. MSE = (1/N)Σ(ŷ-y)² is the negative log-likelihood under `y|x ~ N(ŷ, σ²)`. Minimizing MSE assumes that prediction errors are normally distributed around 0 with equal variance everywhere. This may not be true (luxury homes have larger absolute errors), which is why log-price regression or robust losses sometimes work better.

</details>

---

## 📌 Lab Summary

| What we built | Method | Key result |
|--------------|--------|------------|
| House price model | Linear regression | 2 weights: sqft, bedrooms |
| Loss function | MSE = `(1/N)‖Xw−y‖²` | Scalar measure of prediction quality |
| Gradient | `(2/N)X.T@(Xw−y)` | Direction to improve weights |
| Gradient check | Finite differences ε=1e-5 | Error < 1e-8 → formula is correct |
| Learning rate 0.001 | Too slow | Barely converges in 60 steps |
| Learning rate 0.01 | Slow but stable | Partial convergence |
| Learning rate 0.1 | **Best** | Converges quickly |
| Learning rate 1.0 | **Diverges** | Loss explodes |

---

**🎉 Congratulations — you've completed Week 2!**

You now understand the mathematics behind:
- How neural networks compute gradients (calculus + chain rule)
- How they learn (gradient descent)
- Why they sometimes struggle (saddle points, bad learning rates)
- What loss functions really mean (probability theory + MLE)
- How regularization works (MAP = MLE + prior)

**Next Week:** Linear Algebra meets ML — PCA, SVD, and why matrix factorization is everywhere.